# Phase 6.3.3 — Evidence-aware Structural Reasoning Pilot
Six fixed C_expanded cases; same SAP chunks, library, model and API settings. Revised system prompt and output protocol. Maximum 6 new attempts, no retries. Use a single Colab session. Persistent cache must not be deleted or renamed. Development evaluation only; no DOCX generation.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
ROOT=Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]','openai'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent:',tfl_agent.__file__)
subprocess.run([sys.executable,'-m','pytest','-q',
                str(ROOT/'tests/test_phase6_applicability_pilot.py'),
                str(ROOT/'tests/test_phase6_evidence_applicability.py'),
                str(ROOT/'tests/test_phase6_decision_replay.py'),
                str(ROOT/'tests/test_phase6_llm_pilot.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent.agent.phase6_decision_replay import ARCHIVE_NAME, ARCHIVE_SHA, _sha
from tfl_agent.agent.phase6_evidence_applicability import REPLAY_NAME, REPLAY_SHA
from tfl_agent.agent.phase6_applicability_pilot import (
    AUDIT_NAME,AUDIT_SHA,prepare,run_pilot,export,BUDGET,MODEL)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
DEST=BASE/'Phase633_Applicability_Pilot_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
CACHE=DEST/'Frozen_Applicability_Cache_v1'
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
def frozen_file(folder,name,sha):
    p=BASE/folder/name
    if p.is_file() and _sha(p)==sha:
        return p
    matches=[x for x in p.parent.glob('*.zip') if _sha(x)==sha]
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected {p} with SHA256 {sha}; matches: {matches}')
    return matches[0]
PILOT=frozen_file('Phase63_LLM_Pilot_Outputs',ARCHIVE_NAME,ARCHIVE_SHA)
REPLAY=frozen_file('Phase631_Decision_Replay_Outputs',REPLAY_NAME,REPLAY_SHA)
AUDIT=frozen_file('Phase632_Evidence_Applicability_Outputs',AUDIT_NAME,AUDIT_SHA)
plan=prepare(PILOT,REPLAY,AUDIT)
print('Model:',MODEL,'| new attempt cap:',BUDGET)
print('Persistent cache:',CACHE)
for r in plan['requests']:
    print(r['request_id'], '| chunks:',len(r['payload']['sap_evidence']), '| templates:',len(r['payload']['library']['templates']))
print('Already reserved:',len(list(CACHE.glob('*.attempt.json'))))

## API credentials and authorization
The user approved up to six new DeepSeek attempts in the development conversation. The execution checkbox below confirms authorization to transmit the same SAP passages to DeepSeek. Set DEEPSEEK_API_KEY in Colab Secrets (or enter it privately at the prompt). Never print or save the key. A failed request consumes its reservation; re-running this cell skips all reserved requests. Do not run concurrent sessions.

In [ ]:
from google.colab import userdata
from getpass import getpass
AUTHORIZE_SIX_ATTEMPTS = True #@param {type:"boolean"}
if not AUTHORIZE_SIX_ATTEMPTS:
    raise RuntimeError('Authorization is required for new requests.')
try:
    key=userdata.get('DEEPSEEK_API_KEY')
except Exception:
    key=None
if not key:
    key=getpass('DeepSeek API key (hidden): ')
if not key:
    raise ValueError('Missing API key')
try:
    run_pilot(plan,CACHE,key,authorized=AUTHORIZE_SIX_ATTEMPTS)
finally:
    del key

In [ ]:
import pandas as pd
from IPython.display import display
OUT=Path('/content/Phase633_Applicability_Pilot')/RUN_ID
report,rows=export(plan,CACHE,OUT)
assert report['new_attempts_reserved']<=6
assert len(rows)==6 and not report['generation_authorized']
metrics=[]
for arm,m in report['metrics'].items():
    metrics.append({'arm':arm,'n':m['n'],'raw_top1':m['raw_top1'],
                    'MRR':m['raw_mrr'],'correct_recommendations':m['recommendation']['correct'],
                    'correct_supported':m['support']['correct'],'incorrect_supported':m['support']['incorrect'],
                    'support_coverage':m['support']['coverage'],'support_accuracy':m['support']['accuracy_selected'],
                    'status_counts':m['status_counts']})
display(pd.DataFrame(metrics))
display(pd.DataFrame(rows)[['study_id','entry_id','expected','raw_top','recommended_template',
                            'supported_template','status','error']])
print('New reserved attempts:',report['new_attempts_reserved'])
print('Recorded usage:',report['new_recorded_usage'])
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:',sorted(p.name for p in OUT.iterdir()))
print('Share only if authorized to disclose SAP excerpts and responses.')

## Interpretation
Compare D_applicability primarily against C_expanded on the same six cases. Report protocol failure, raw ranking, support accuracy and coverage separately. Development_Reference_Review.csv contains ten pre-pilot qualitative reference notes, not independent clinical gold. Overlapping citations are review aids only; missing overlap is not necessarily an omission. Revised_Citation_Review.csv requires semantic review before claiming reduced evidence misuse. Model-declared applicability and structural support are not independently verified. No new calls beyond the six reservations; no generation authorization.